In [5]:
import pycaret
import mlflow

print("=======================================")
print(f"Versi PyCaret yang terinstall: {pycaret.__version__}")
print(f"Versi MLflow yang terinstall : {mlflow.__version__}")
print("=======================================")

Versi PyCaret yang terinstall: 3.3.2
Versi MLflow yang terinstall : 2.14.1


In [1]:
import pandas as pd
from pycaret.regression import setup, compare_models, evaluate_model,tune_model, finalize_model, predict_model, pull


In [2]:
data = pd.read_csv(r"D:\Sem 4\PBL\data\processed\data_proses_mingguan.csv")
data.head()

,Order Date,Quantity,Sales,Profit,Discount,Quantity_Lag_1,Quantity_Lag_2,Quantity_Rolling_Mean_3,Sales_Lag_1,Profit_Lag_1,Month,Year,Week_of_Year,Sub-Category
0,2014-01-26,0,0.00,0.00,0.00,11.0,3.0,6.666667,796.69,324.68,1,2014,4,Accessories
1,2014-02-02,0,0.00,0.00,0.00,0.0,11.0,4.666667,0.00,0.00,2,2014,5,Accessories
2,2014-02-09,15,888.32,190.43,0.15,0.0,0.0,3.666667,0.00,0.00,2,2014,6,Accessories
3,2014-02-16,24,1292.76,409.84,0.10,15.0,0.0,5.000000,888.32,190.43,2,2014,7,Accessories
4,2014-02-23,3,62.31,22.43,0.00,24.0,15.0,13.000000,1292.76,409.84,2,2014,8,Accessories


In [3]:
# Pilihan: Ambil daftar sub-kategori unik dari data mingguan
sub_categories = data['Sub-Category'].unique()

print("Memulai Pemodelan Otomatis dengan Tracking MLflow...")


Memulai Pemodelan Otomatis dengan Tracking MLflow...


In [8]:
# 1. Wadah untuk dokumentasi top 3 baseline model per sub-kategori
tabel_dokumentasi = []

# Ambil daftar sub-kategori unik dari data mingguan
sub_categories = data['Sub-Category'].unique()

print("Memulai Pemodelan Otomatis dengan Tracking MLflow...")

# Looping otomatis untuk setiap sub-kategori
for sub_cat in sub_categories:
    print(f"\n=======================================================")
    print(f"[PROSES] Menghitung Model untuk Sub-Kategori: {sub_cat}")
    print(f"=======================================================")
    
    # Filter data per sub-kategori
    df_sub = data[data['Sub-Category'] == sub_cat].copy()
    
    # Proteksi: Lewati jika data terlalu sedikit untuk di-split train-test
    if len(df_sub) < 10:
        print(f"[LEWAT] Sub-kategori {sub_cat} dilewati karena data terlalu sedikit ({len(df_sub)} baris).")
        continue
    
    # Drop kolom data bocor / teks mentah
    cols_to_drop = ['Order Date', 'Sales', 'Profit', 'Sub-Category']
    X_data = df_sub.drop(columns=[col for col in cols_to_drop if col in df_sub.columns], errors='ignore')
    
    # 2. Setup PyCaret + Otomatis Log ke MLflow
    # Kita tambahkan log_plots=True sebagai pengganti evaluate_model() agar tidak membuat notebook lag
    grid = setup(
        data=X_data, 
        target='Quantity', 
        session_id=42,
        log_experiment=True,                       # AKTIFKAN LOG OTOMATIS MLflow
        experiment_name=f"Weekly_{sub_cat}",       # Nama Eksperimen di MLflow UI
        log_plots=True,                            # Otomatis mengirimkan plot evaluasi ke MLflow
        verbose=False,                             
        html=False
    )
    
    # 3. Cari Top 3 Model Terbaik berdasarkan MAE (Exclude CatBoost demi keamanan bug clone)
    top3_models = compare_models(n_select=3, sort='MAE', exclude=['catboost'], verbose=False)
    print(f"[SUKSES] Top 3 Model untuk {sub_cat} telah otomatis tercatat di MLflow.")

    # Ambil papan skor metrik menggunakan pull()
    leaderboard = pull() 
    top3_metrics = leaderboard.head(3)
    
    # Masukkan baris demi baris hasil baseline ke dalam list dokumentasi
    for urutan, (nama_model_baseline, baris_metrik) in enumerate(top3_metrics.iterrows(), 1):
        tabel_dokumentasi.append({
            'Sub-Category': sub_cat,
            'Rank': f"Top {urutan}",
            'Model Name': nama_model_baseline,
            'MAE': baris_metrik['MAE'],
            'RMSE': baris_metrik['RMSE'],
            'R2': baris_metrik['R2'],
            'MAPE': baris_metrik['MAPE']
        })

    # 4. Ambil model peringkat 1 sebagai default awal untuk proses selanjutnya
    model_terpilih = top3_models[0] 
    nama_model = type(model_terpilih).__name__ 
    
    # =========================================================================
    # BLOK PROTEKSI: TUNING MODEL TOP 1
    # =========================================================================
    try:
        print(f"-> Mencoba melakukan tuning pada model: {nama_model}...")
        tuned_model = tune_model(model_terpilih, optimize='MAE', verbose=False)
        
        # Jika sukses, variabel model_terpilih diperbarui menjadi versi tuning
        model_terpilih = tuned_model
        print(f"[SUKSES] Model {nama_model} BERHASIL di-tuning!")
        
    except Exception as e:
        print(f"[PERINGATAN] Model {nama_model} tidak mendukung otomatis tuning (Error Clone).")
        print(f"-> Solusi Otomatis: Tetap menggunakan model {nama_model} versi ASLI (Tanpa Tuning).")

    # =========================================================================
    # URUTAN YANG BENAR: PREDICT/EVALUASI DULU -> BARU FINALIZE
    # =========================================================================
    # a. Predict Model (Evaluasi pada data Holdout / Test set)
    # Langkah ini akan otomatis mengirimkan skor performa data uji ke MLflow
    predict_model(model_terpilih, verbose=False)
    print(f"[SUKSES] Prediksi & Holdout Metrics untuk {sub_cat} telah tercatat di MLflow.")

    # b. Evaluasi Model (Tampilkan plot evaluasi di notebook dan kirim ke MLflow)
    # Langkah ini akan otomatis mengirimkan plot evaluasi ke MLflow
    evaluate_model(model_terpilih)
    print(f"[SUKSES] Evaluasi Model untuk {sub_cat} telah tercatat di MLflow.")

    # c. Finalize Model (Latihan ulang dengan 100% gabungan data Train + Test)
    final_model = finalize_model(model_terpilih)
    print(f"[SUKSES] Model terbaik untuk {sub_cat} telah difinalisasi dan siap dipakai forecasting.")

# =========================================================================
# KELUAR DARI LOOPING: PROSES AKHIR DOKUMENTASI CSV
# =========================================================================
print("\n=======================================================")
print("[PROSES SELESAI] Membuat DataFrame dan Menyimpan CSV...")
print("=======================================================")

# Mengubah list kumpulan data tadi menjadi DataFrame Pandas yang rapi
df_hasil_eksperimen = pd.DataFrame(tabel_dokumentasi)

# Menyimpan DataFrame menjadi file CSV di folder projekmu
df_hasil_eksperimen.to_csv('dokumentasi_baseline_models_weekly.csv', index=False)

print("✅ File 'dokumentasi_baseline_models_weekly.csv' berhasil dibuat!")
print("👉 Silakan buka terminal dan ketik: 'mlflow ui' untuk melihat visualisasinya.")
print("=======================================================")

Memulai Pemodelan Otomatis dengan Tracking MLflow...

[PROSES] Menghitung Model untuk Sub-Kategori: Accessories
[SUKSES] Top 3 Model untuk Accessories telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: GradientBoostingRegressor...
[SUKSES] Model GradientBoostingRegressor BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Accessories telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Accessories telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Accessories telah difinalisasi dan siap dipakai forecasting.

[PROSES] Menghitung Model untuk Sub-Kategori: Appliances
[SUKSES] Top 3 Model untuk Appliances telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: RandomForestRegressor...
[SUKSES] Model RandomForestRegressor BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Appliances telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Appliances telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Appliances telah difinalisasi dan siap dipakai forecasting.

[PROSES] Menghitung Model untuk Sub-Kategori: Art
[SUKSES] Top 3 Model untuk Art telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: GradientBoostingRegressor...
[SUKSES] Model GradientBoostingRegressor BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Art telah tercatat di MLflow.


...

[SUKSES] Evaluasi Model untuk Storage telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Storage telah difinalisasi dan siap dipakai forecasting.

[PROSES] Menghitung Model untuk Sub-Kategori: Supplies
[SUKSES] Top 3 Model untuk Supplies telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: Lars...
[SUKSES] Model Lars BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Supplies telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Supplies telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Supplies telah difinalisasi dan siap dipakai forecasting.

[PROSES] Menghitung Model untuk Sub-Kategori: Tables
[SUKSES] Top 3 Model untuk Tables telah otomatis tercatat di MLflow.
-> Mencoba melakukan tuning pada model: RandomForestRegressor...
[SUKSES] Model RandomForestRegressor BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Tables telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Tables telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Tables telah difinalisasi dan siap dipakai forecasting.

[PROSES SELESAI] Membuat DataFrame dan Menyimpan CSV...
✅ File 'dokumentasi_baseline_models_weekly.csv' berhasil dibuat!
👉 Silakan buka terminal dan ketik: 'mlflow ui' untuk melihat visualisasinya.


In [7]:
import pandas as pd
import os
from pycaret.regression import setup, compare_models, pull

# =========================================================================
# 1. AMBIL DATA MINGGUAN KHUSUS UNTUK 3 PRODUK FLUKTUATIF
# =========================================================================
# Silakan ganti 'df_weekly_master' dengan nama DataFrame mingguanmu
df_weekly = data.copy() 

# Daftar produk yang ingin kita jinakkan dengan model linear
sub_categories_fluktuatif = ['Appliances', 'Paper', 'Furnishings']

# Wadah baru untuk menampung dokumentasi khusus model linear
tabel_dokumentasi_linear = []

print("Memulai Pemodelan khusus (MODEL LINEAR) untuk Sub-Kategori Fluktuatif...")

for sub_cat in sub_categories_fluktuatif:
    print(f"\n=======================================================")
    # Kita beri penanda khusus di print log
    print(f"[PROSES LINEAR] Menghitung Model untuk Sub-Kategori: {sub_cat}")
    print(f"=======================================================")
    
    # Filter data mingguan per sub-kategori
    df_sub = df_weekly[df_weekly['Sub-Category'] == sub_cat].copy()
    
    # Proteksi jumlah data
    if len(df_sub) < 6:
        print(f"[LEWAT] Data {sub_cat} terlalu sedikit.")
        continue
    
    # Drop kolom data bocor / teks mentah
    cols_to_drop = ['Order Date', 'Sales', 'Profit', 'Sub-Category']
    X_data = df_sub.drop(columns=[col for col in cols_to_drop if col in df_sub.columns], errors='ignore')
    
    # 2. Setup PyCaret 
    grid = setup(
        data=X_data, 
        target='Quantity', 
        session_id=42,
        log_experiment=True, 
        experiment_name=f"Weekly_Linear_{sub_cat}", # Nama eksperimen khusus di MLflow
        log_plots=True,
        verbose=False, 
        html=False
    )
    
    # 3. KUNCI HANYA UNTUK MODEL LINEAR (include parameter diaktifkan)
    top3_models = compare_models(
        n_select=3,
        include=['lr', 'lasso', 'ridge', 'en', 'huber'], # Hanya mengecek 5 model linear ini
        sort='MAE', 
        verbose=False
    )
    print(f"[SUKSES] Top 3 Model Linear untuk {sub_cat} tercatat di MLflow.")

    # Ambil papan skor metrik
    leaderboard = pull() 
    top3_metrics = leaderboard.head(3)
    
    # Masukkan ke dalam list dokumentasi baris demi baris
    for urutan, (nama_model_baseline, baris_metrik) in enumerate(top3_metrics.iterrows(), 1):
        tabel_dokumentasi_linear.append({
            'Sub-Category': sub_cat,
            'Rank': f"Top {urutan} (Linear)", # Diberi label (Linear) agar mudah dibedakan di CSV
            'Model Name': nama_model_baseline,
            'MAE': baris_metrik['MAE'],
            'RMSE': baris_metrik['RMSE'],
            'R2': baris_metrik['R2'],
            'MAPE': baris_metrik['MAPE']
        })
    # 4. Ambil model peringkat 1 sebagai default awal untuk proses selanjutnya
    model_terpilih = top3_models[0] 
    nama_model = type(model_terpilih).__name__ 
    
    # =========================================================================
    # BLOK PROTEKSI: TUNING MODEL TOP 1
    # =========================================================================
    try:
        print(f"-> Mencoba melakukan tuning pada model: {nama_model}...")
        tuned_model = tune_model(model_terpilih, optimize='MAE', verbose=False)
        
        # Jika sukses, variabel model_terpilih diperbarui menjadi versi tuning
        model_terpilih = tuned_model
        print(f"[SUKSES] Model {nama_model} BERHASIL di-tuning!")
        
    except Exception as e:
        print(f"[PERINGATAN] Model {nama_model} tidak mendukung otomatis tuning (Error Clone).")
        print(f"-> Solusi Otomatis: Tetap menggunakan model {nama_model} versi ASLI (Tanpa Tuning).")

    # =========================================================================
    # URUTAN YANG BENAR: PREDICT/EVALUASI DULU -> BARU FINALIZE
    # =========================================================================
    # a. Predict Model (Evaluasi pada data Holdout / Test set)
    # Langkah ini akan otomatis mengirimkan skor performa data uji ke MLflow
    predict_model(model_terpilih, verbose=False)
    print(f"[SUKSES] Prediksi & Holdout Metrics untuk {sub_cat} telah tercatat di MLflow.")

    # b. Evaluasi Model (Tampilkan plot evaluasi di notebook dan kirim ke MLflow)
    # Langkah ini akan otomatis mengirimkan plot evaluasi ke MLflow
    evaluate_model(model_terpilih)
    print(f"[SUKSES] Evaluasi Model untuk {sub_cat} telah tercatat di MLflow.")

    # c. Finalize Model (Latihan ulang dengan 100% gabungan data Train + Test)
    final_model = finalize_model(model_terpilih)
    print(f"[SUKSES] Model terbaik untuk {sub_cat} telah difinalisasi dan siap dipakai forecasting.")

# Mengubah hasil linear baru menjadi DataFrame
df_hasil_linear_baru = pd.DataFrame(tabel_dokumentasi_linear)

# =========================================================================
# 4. PROSES MERGE/PENGGABUNGAN KE CSV YANG SAMA
# =========================================================================
nama_file_csv = 'dokumentasi_baseline_models_weekly.csv'

if os.path.exists(nama_file_csv):
    print("\n[MENGGABUNGKAN] Membaca file dokumentasi mingguan yang lama...")
    df_lama = pd.read_csv(nama_file_csv)
    
    # LANGKAH PENTING: Hapus data 3 produk lama yang berbasis pohon (karena hasilnya minus)
    # Kebalikan dari isin (menggunakan tanda ~) artinya "Ambil yang BUKAN Appliances, Paper, Furnishings"
    df_lama_bersih = df_lama[~df_lama['Sub-Category'].isin(sub_categories_fluktuatif)]
    
    # Gabungkan data lama yang sudah bersih dengan hasil model linear kita yang baru
    df_final_dokumentasi = pd.concat([df_lama_bersih, df_hasil_linear_baru], ignore_index=True)
    print("-> Berhasil menimpa data 3 produk fluktuatif dengan versi Model Linear.")
else:
    # Jika file lama belum ada/terhapus, otomatis membuat file baru dari hasil linear ini
    df_final_dokumentasi = df_hasil_linear_baru
    print("\n[PERINGATAN] File lama tidak ditemukan. Membuat file dokumentasi baru.")

# Simpan kembali hasil gabungan final ke file yang sama
df_final_dokumentasi.to_csv(nama_file_csv, index=False)

print("\n=======================================================")
print(f"✅ HORE! File '{nama_file_csv}' berhasil diperbarui dengan aman!")
print("Sekarang file tersebut berisi kombinasi Model Pohon & Model Linear pilihanmu.")
print("=======================================================")

Memulai Pemodelan khusus (MODEL LINEAR) untuk Sub-Kategori Fluktuatif...

[PROSES LINEAR] Menghitung Model untuk Sub-Kategori: Appliances
[SUKSES] Top 3 Model Linear untuk Appliances tercatat di MLflow.
-> Mencoba melakukan tuning pada model: LinearRegression...
[SUKSES] Model LinearRegression BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Appliances telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Appliances telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Appliances telah difinalisasi dan siap dipakai forecasting.

[PROSES LINEAR] Menghitung Model untuk Sub-Kategori: Paper
[SUKSES] Top 3 Model Linear untuk Paper tercatat di MLflow.
-> Mencoba melakukan tuning pada model: Lasso...
[SUKSES] Model Lasso BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Paper telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Paper telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Paper telah difinalisasi dan siap dipakai forecasting.

[PROSES LINEAR] Menghitung Model untuk Sub-Kategori: Furnishings
[SUKSES] Top 3 Model Linear untuk Furnishings tercatat di MLflow.
-> Mencoba melakukan tuning pada model: Lasso...
[SUKSES] Model Lasso BERHASIL di-tuning!
[SUKSES] Prediksi & Holdout Metrics untuk Furnishings telah tercatat di MLflow.


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

[SUKSES] Evaluasi Model untuk Furnishings telah tercatat di MLflow.
[SUKSES] Model terbaik untuk Furnishings telah difinalisasi dan siap dipakai forecasting.

[MENGGABUNGKAN] Membaca file dokumentasi mingguan yang lama...
-> Berhasil menimpa data 3 produk fluktuatif dengan versi Model Linear.

✅ HORE! File 'dokumentasi_baseline_models_weekly.csv' berhasil diperbarui dengan aman!
Sekarang file tersebut berisi kombinasi Model Pohon & Model Linear pilihanmu.
